In [2]:
import os
import numpy as np
from util import *
import random
import gc
import copy as c
import pickle
import json
from sklearn.metrics import precision_recall_curve, log_loss, auc, precision_score, recall_score, f1_score
from scipy.signal import argrelextrema
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tabulate import tabulate
import torch

[   INFO   ] MusicExtractorSVM: no classifier models were configured by default


In [3]:
batch_size = 32
max_chunk_size = 1000
classification_scheme = 'multilabel'
memlen = 15
nframes = 32
nmelbands = 80
nchannels = 3
name_from_fp = lambda x: os.path.splitext(os.path.split(x)[1])[0]

lbl_fp = 'onset/songs/stream_labels.pkl'
vld_fp = 'onset/songs/songs_valid.txt'
json_fp = 'json/songs/songs_valid.txt'

ddc_memlen = 100
context_radius = 7
ddc_npred_steps = 5
ddc_tolerance = 2
ddc_removal_order = [1,-1,2,-2,3,-3]
feats_dir = 'feats/songs'
diff_dict = {
    'Beginner': 0,
    'Easy': 1,
    'Medium': 2,
    'Hard': 3,
    'Edit': 4,
    'Challenge': 4
}
metrics = ['f1_coarse', 'f1_fine', 'f1_avg',
           'max_f1_coarse', 'max_f1_fine', 'max_f1_avg',
           'auc_coarse', 'auc_fine',
           'prec_coarse', 'prec_fine', 'prec_avg',
           'max_prec_coarse', 'max_prec_fine', 'max_prec_avg',
           'recall_coarse', 'recall_fine', 'recall_avg',
           'max_recall_coarse', 'max_recall_fine', 'max_recall_avg',
           'log_loss_coarse', 'log_loss_fine', 
           'max_thresh_coarse', 'max_thresh_fine', 'max_thresh_avg']

In [4]:
# Load the saved dictionary
metrics_save_path = 'onset_keras_metrics.pkl'

with open(metrics_save_path, 'rb') as f:
    keras_metric_dict = pickle.load(f)

print("Successfully loaded Keras metric_dict! Ready for comparison tables.")
# You can now reference `keras_metric_dict` in your downstream table 
print(keras_metric_dict.keys())

Successfully loaded Keras metric_dict! Ready for comparison tables.
dict_keys(['ddc', 'ddcl'])


In [25]:
goct_metrics_save_path = 'metrics_beatfine_fraxtil_timingonly.pkl'
with open(goct_metrics_save_path, 'rb') as f:
    goct_metric_dict = pickle.load(f)

print("Successfully loaded goct metric_dict! Ready for comparison tables.")
# You can now reference `keras_metric_dict` in your downstream table 
goct_metric_dict = {'max_' + k: v for k,v in zip(goct_metric_dict.keys(), goct_metric_dict.values())}
print(goct_metric_dict['max_f1_coarse'])


Successfully loaded goct metric_dict! Ready for comparison tables.
{11: [0.733433734939759, 0.8322756119673618, 0.6479357798165138, 0.8973185088293002, 0.7310061601642711, 0.8981233243967829, 0.7412040656763096, 0.8674431503050473, 0.8218298555377207, 0.6886858749121575, 0.5673306772908365], 14: [0.6438515081206496], 15: [0.786174575278266, 0.9524594668865072], 1: [0.7244897959183674, 0.6962962962962963, 0.8750000000000001, 0.8227848101265823, 0.6105263157894736, 0.6909090909090909, 0.8717948717948718, 0.4444444444444444, 0.7384615384615386, 0.7012987012987013, 0.33333333333333326], 3: [0.7319884726224782, 0.6036960985626283, 0.8356807511737089, 0.8238213399503722, 0.859180035650624, 0.8904428904428905, 0.7927927927927928, 0.8264840182648402, 0.7738419618528609], 9: [0.6514181152790485, 0.7642276422764228, 0.6833631484794275, 0.7579710144927536, 0.7194860813704497, 0.86], 5: [0.8024948024948024, 0.7142857142857143, 0.7859007832898173, 0.8900709219858156, 0.8548707753479126], 12: [0.82,

In [7]:
from onset_cleaned import *
# 1. Recreate the precise configuration used during training
config = OnsetConfig(
    d_model=256,       # Default matching parser.add_argument("--d_model", default=256)
    n_enc_layers=4,    # Default matching parser.add_argument("--enc_layers", default=4)
    n_dec_layers=4     # Default matching parser.add_argument("--dec_layers", default=4)
)

# 2. Instantiate the model structural architecture
onset_model = OnsetModel(config)

# 3. Resolve the path to your best checkpoint file
# Matches: os.path.join(model_dir, f"{model_name}_best.pt")
model_dir = "trained_models"
model_name = "ddr_onset_v1"
checkpoint_path = os.path.join(model_dir, f"{model_name}_hier_gate_best_f1.pt")

# 4. Load the weights state dictionary into the model
if os.path.exists(checkpoint_path):
    import numpy as np
    torch.serialization.add_safe_globals([np.core.multiarray.scalar])
    
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'), weights_only=False)
    
    # Extract the nested model weight weights dictionary based on the traceback fields
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        onset_model.load_state_dict(checkpoint['model_state'])
    elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        onset_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        onset_model.load_state_dict(checkpoint)
        
    print(f"Successfully loaded best checkpoint weights from: {checkpoint_path}")
    if isinstance(checkpoint, dict) and 'epoch' in checkpoint:
        print(f"Checkpoint info -> Saved at Epoch: {checkpoint['epoch']} | Val PR AUC: {checkpoint.get('val_pr_auc', 'N/A')}")
else:
    raise FileNotFoundError(f"Could not find a checkpoint file at {checkpoint_path}.")

Successfully loaded best checkpoint weights from: trained_models/ddr_onset_v1_hier_gate_best_f1.pt
Checkpoint info -> Saved at Epoch: 31 | Val PR AUC: 0.8600120168944906


/tmp/ipykernel_3013561/2494297868.py:21: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  torch.serialization.add_safe_globals([np.core.multiarray.scalar])


In [8]:
# Ensure model is ready and on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
onset_model.to(device)
onset_model.eval()

# 1. Initialize Tracking Structures matching your layout
metric_dict = {name: dict() for name in metrics}

vld_ds = get_dataset_fp_list(vld_fp)

tp, fp, fn = 0, 0, 0
mtp, mfp, mfn = 0, 0, 0
threshlist = []

with open(lbl_fp, 'rb') as f:
    labels = pickle.load(f)
enc_dict = label_to_vect_dict(labels)

# 2. Sequential Validation Pass over the dataset
print(f"Starting non-autoregressive one-shot validation pass over {len(vld_ds)} files...")

for fpt in vld_ds:
    with open(fpt, 'rb') as f:
        charts = pickle.load(f)
        
    feats_fp = charts[3]  # Extract independent feature reference path
    with open(feats_fp, 'rb') as f:
        feats = pickle.load(f)
        
    # Step through every chart variant contained inside this entry
    for j in range(len(charts[0])):
        chart = [charts[i][j] for i in range(3)]
        
        diff_coarse = diff_dict[chart[1][0][-1]]
        diff_fine = chart[1][0][0]
        
        # Safely capture the scalar BPM sequence value before chart[1] is modified
        bpm_val = chart[1][0][1]

        # Resolve features from external file via window context ranges
        chart[0] = [make_onset_feature_context_range(feats, x[0], x[1]) for x in chart[0]]
        
        # Standardize features
        mean = np.mean(chart[0], axis=0)
        std = np.std(chart[0], axis=0)
        chart[0] = (np.array(chart[0]) - mean) / (std + 1e-7)
        
        chart[1] = [[a[0], a[1]] for a in chart[1]]
        placed_notes = np.array([enc_dict[i] for i in chart[2]])

        # ------------------------------------------------------------------
        # FIXED: One-Shot Score Extraction & Sigmoid Conversion
        # ------------------------------------------------------------------
        # Input shape expected by OnsetModel: (Batch=1, Time, Frames, Freq, Channels)
        feats_t = torch.tensor(chart[0], dtype=torch.float32).unsqueeze(0).to(device)
        
        bpm_t = torch.tensor([float(bpm_val)], dtype=torch.float32).to(device)
        diff_t = torch.tensor([int(diff_fine)], dtype=torch.long).to(device)
        
        with torch.no_grad():
            # Use predict_prob to ensure values are sigmoided probabilities bounded [0, 1]
            # or extract forward and apply torch.sigmoid directly.
            scores_t = onset_model.predict_prob(feats_t, bpm_t, diff_t)
            
            # Squeeze batch dimension and transfer to CPU NumPy array: shape (T, 48)
            scores = scores_t.squeeze(0).cpu().numpy()
        # ------------------------------------------------------------------
        
        # 3. Micro-level Multi-Label Metrics Extraction
        flat_placed = placed_notes.flatten()
        flat_scores = scores.flatten()
        
        prec, rec, threshes = precision_recall_curve(flat_placed, flat_scores)
        AUC = auc(rec, prec)
        
        # safely bound log_loss predictions using clip to avoid stability blowups at 0 or 1
        flat_scores_clipped = np.clip(flat_scores, 1e-7, 1 - 1e-7)
        loss = log_loss(flat_placed, flat_scores_clipped, labels=[0, 1])
        
        fd = prec + rec
        fd[np.where(fd == 0)] = 1
        fs = (2 * (prec * rec)) / fd
        MID = np.argmax(fs)
        
        # Prevent index errors if precision_recall_curve returns empty boundary arrays
        if len(threshes) > 0 and MID < len(threshes):
            MaxF1, MaxPrec, MaxRec, MaxThresh = fs[MID], prec[MID], rec[MID], threshes[MID]
        else:
            MaxF1, MaxPrec, MaxRec, MaxThresh = fs[MID], prec[MID], rec[MID], 0.5
            
        threshlist.append(MaxThresh)
        
        # Accumulate metrics based on dynamically found optimal threshold
        mtp += np.sum((flat_scores >= MaxThresh) & (flat_placed == 1))
        mfp += np.sum((flat_scores >= MaxThresh) & (flat_placed == 0))
        mfn += np.sum((flat_scores < MaxThresh) & (flat_placed == 1))

        chart_tp = np.sum((flat_scores >= 0.5) & (flat_placed == 1))
        chart_fp = np.sum((flat_scores >= 0.5) & (flat_placed == 0))
        chart_fn = np.sum((flat_scores < 0.5) & (flat_placed == 1))
        
        # Global accumulation trackers
        tp += chart_tp
        fp += chart_fp
        fn += chart_fn

        # Calculate exact chart-level metrics via the counting method
        F1 = chart_tp / (chart_tp + 0.5 * (chart_fp + chart_fn)) if (chart_tp + chart_fp + chart_fn) > 0 else 0
        Prec = chart_tp / (chart_tp + chart_fp) if (chart_tp + chart_fp) > 0 else 0
        Recall = chart_tp / (chart_tp + chart_fn) if (chart_tp + chart_fn) > 0 else 0

        out_met = {
            'log_loss': loss, 'f1': F1, 'auc': AUC, 'prec': Prec, 'recall': Recall, 
            'max_f1': MaxF1, 'max_prec': MaxPrec, 'max_recall': MaxRec, 'max_thresh': MaxThresh
        }
        
        # Populate categorized nested dictionaries
        for k, v in out_met.items():
            for categorization, diff_key in [('_coarse', diff_coarse), ('_fine', diff_fine)]:
                kc = k + categorization
                if kc in metric_dict:
                    if diff_key in metric_dict[kc]:
                        metric_dict[kc][diff_key].append(v)
                    else:
                        metric_dict[kc][diff_key] = [v]
                        
    del feats, charts
    gc.collect()

# 5. Global Metrics Summarization
F1 = tp / (tp + 0.5 * (fp + fn)) if (tp + fp + fn) > 0 else 0
Prec = tp / (tp + fp) if (tp + fp) > 0 else 0
Recall = tp / (tp + fn) if (tp + fn) > 0 else 0   
MaxF1 = mtp / (mtp + 0.5 * (mfp + mfn)) if (mtp + mfp + mfn) > 0 else 0
MaxPrec = mtp / (mtp + mfp) if (mtp + mfp) > 0 else 0
MaxRecall = mtp / (mtp + mfn) if (mtp + mfn) > 0 else 0
MaxThresh = np.median(threshlist) if threshlist else 0

metric_dict['f1_avg'] = F1
metric_dict['prec_avg'] = Prec
metric_dict['recall_avg'] = Recall
metric_dict['max_f1_avg'] = MaxF1
metric_dict['max_prec_avg'] = MaxPrec
metric_dict['max_recall_avg'] = MaxRecall
metric_dict['max_thresh_avg'] = MaxThresh

print("\n--- Final Results Summary ---")
print(f'f1 avg: {F1:.4f}\nPrec avg: {Prec:.4f}\nRecall avg: {Recall:.4f}')
print(f'Max f1 avg: {MaxF1:.4f}\nMax Prec avg: {MaxPrec:.4f}\nMax Recall avg: {MaxRecall:.4f}')
print(f'Max Thresh median: {MaxThresh:.4f}')

ITGPT_onset = metric_dict

Starting non-autoregressive one-shot validation pass over 24 files...

--- Final Results Summary ---
f1 avg: 0.7801
Prec avg: 0.7538
Recall avg: 0.8084
Max f1 avg: 0.8022
Max Prec avg: 0.7621
Max Recall avg: 0.8467
Max Thresh median: 0.4156


In [9]:
import onset_nohier as nohier
# 1. Recreate the precise configuration used during training
config = OnsetConfig(
    d_model=256,       # Default matching parser.add_argument("--d_model", default=256)
    n_enc_layers=4,    # Default matching parser.add_argument("--enc_layers", default=4)
    n_dec_layers=4     # Default matching parser.add_argument("--dec_layers", default=4)
)

# 2. Instantiate the model structural architecture
onset_model = nohier.OnsetModel(config)

# 3. Resolve the path to your best checkpoint file
# Matches: os.path.join(model_dir, f"{model_name}_best.pt")
model_dir = "trained_models"
model_name = "ddr_onset_v1"
checkpoint_path = os.path.join(model_dir, f"{model_name}_nohier_best_f1.pt")

# 4. Load the weights state dictionary into the model
if os.path.exists(checkpoint_path):
    import numpy as np
    torch.serialization.add_safe_globals([np.core.multiarray.scalar])
    
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'), weights_only=False)
    
    # Extract the nested model weight weights dictionary based on the traceback fields
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        onset_model.load_state_dict(checkpoint['model_state'])
    elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        onset_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        onset_model.load_state_dict(checkpoint)
        
    print(f"Successfully loaded best checkpoint weights from: {checkpoint_path}")
    if isinstance(checkpoint, dict) and 'epoch' in checkpoint:
        print(f"Checkpoint info -> Saved at Epoch: {checkpoint['epoch']} | Val PR AUC: {checkpoint.get('val_pr_auc', 'N/A')}")
else:
    raise FileNotFoundError(f"Could not find a checkpoint file at {checkpoint_path}.")

Successfully loaded best checkpoint weights from: trained_models/ddr_onset_v1_nohier_best_f1.pt
Checkpoint info -> Saved at Epoch: 35 | Val PR AUC: 0.8598361855971896


In [10]:
# Ensure model is ready and on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
onset_model.to(device)
onset_model.eval()

# 1. Initialize Tracking Structures matching your layout
metric_dict = {name: dict() for name in metrics}

vld_ds = get_dataset_fp_list(vld_fp)

tp, fp, fn = 0, 0, 0
mtp, mfp, mfn = 0, 0, 0
threshlist = []

with open(lbl_fp, 'rb') as f:
    labels = pickle.load(f)
enc_dict = label_to_vect_dict(labels)

# 2. Sequential Validation Pass over the dataset
print(f"Starting non-autoregressive one-shot validation pass over {len(vld_ds)} files...")

for fpt in vld_ds:
    with open(fpt, 'rb') as f:
        charts = pickle.load(f)
        
    feats_fp = charts[3]  # Extract independent feature reference path
    with open(feats_fp, 'rb') as f:
        feats = pickle.load(f)
        
    # Step through every chart variant contained inside this entry
    for j in range(len(charts[0])):
        chart = [charts[i][j] for i in range(3)]
        
        diff_coarse = diff_dict[chart[1][0][-1]]
        diff_fine = chart[1][0][0]
        
        # Safely capture the scalar BPM sequence value before chart[1] is modified
        bpm_val = chart[1][0][1]

        # Resolve features from external file via window context ranges
        chart[0] = [make_onset_feature_context_range(feats, x[0], x[1]) for x in chart[0]]
        
        # Standardize features
        mean = np.mean(chart[0], axis=0)
        std = np.std(chart[0], axis=0)
        chart[0] = (np.array(chart[0]) - mean) / (std + 1e-7)
        
        chart[1] = [[a[0], a[1]] for a in chart[1]]
        placed_notes = np.array([enc_dict[i] for i in chart[2]])

        # ------------------------------------------------------------------
        # FIXED: One-Shot Score Extraction & Sigmoid Conversion
        # ------------------------------------------------------------------
        # Input shape expected by OnsetModel: (Batch=1, Time, Frames, Freq, Channels)
        feats_t = torch.tensor(chart[0], dtype=torch.float32).unsqueeze(0).to(device)
        
        bpm_t = torch.tensor([float(bpm_val)], dtype=torch.float32).to(device)
        diff_t = torch.tensor([int(diff_fine)], dtype=torch.long).to(device)
        
        with torch.no_grad():
            # Use predict_prob to ensure values are sigmoided probabilities bounded [0, 1]
            # or extract forward and apply torch.sigmoid directly.
            scores_t = onset_model.predict_prob(feats_t, bpm_t, diff_t)
            
            # Squeeze batch dimension and transfer to CPU NumPy array: shape (T, 48)
            scores = scores_t.squeeze(0).cpu().numpy()
        # ------------------------------------------------------------------
        
        # 3. Micro-level Multi-Label Metrics Extraction
        flat_placed = placed_notes.flatten()
        flat_scores = scores.flatten()
        
        prec, rec, threshes = precision_recall_curve(flat_placed, flat_scores)
        AUC = auc(rec, prec)
        
        # safely bound log_loss predictions using clip to avoid stability blowups at 0 or 1
        flat_scores_clipped = np.clip(flat_scores, 1e-7, 1 - 1e-7)
        loss = log_loss(flat_placed, flat_scores_clipped, labels=[0, 1])
        
        fd = prec + rec
        fd[np.where(fd == 0)] = 1
        fs = (2 * (prec * rec)) / fd
        MID = np.argmax(fs)
        
        # Prevent index errors if precision_recall_curve returns empty boundary arrays
        if len(threshes) > 0 and MID < len(threshes):
            MaxF1, MaxPrec, MaxRec, MaxThresh = fs[MID], prec[MID], rec[MID], threshes[MID]
        else:
            MaxF1, MaxPrec, MaxRec, MaxThresh = fs[MID], prec[MID], rec[MID], 0.5
            
        threshlist.append(MaxThresh)
        
        # Accumulate metrics based on dynamically found optimal threshold
        mtp += np.sum((flat_scores >= MaxThresh) & (flat_placed == 1))
        mfp += np.sum((flat_scores >= MaxThresh) & (flat_placed == 0))
        mfn += np.sum((flat_scores < MaxThresh) & (flat_placed == 1))

        chart_tp = np.sum((flat_scores >= 0.5) & (flat_placed == 1))
        chart_fp = np.sum((flat_scores >= 0.5) & (flat_placed == 0))
        chart_fn = np.sum((flat_scores < 0.5) & (flat_placed == 1))
        
        # Global accumulation trackers
        tp += chart_tp
        fp += chart_fp
        fn += chart_fn

        # Calculate exact chart-level metrics via the counting method
        F1 = chart_tp / (chart_tp + 0.5 * (chart_fp + chart_fn)) if (chart_tp + chart_fp + chart_fn) > 0 else 0
        Prec = chart_tp / (chart_tp + chart_fp) if (chart_tp + chart_fp) > 0 else 0
        Recall = chart_tp / (chart_tp + chart_fn) if (chart_tp + chart_fn) > 0 else 0

        out_met = {
            'log_loss': loss, 'f1': F1, 'auc': AUC, 'prec': Prec, 'recall': Recall, 
            'max_f1': MaxF1, 'max_prec': MaxPrec, 'max_recall': MaxRec, 'max_thresh': MaxThresh
        }
        
        # Populate categorized nested dictionaries
        for k, v in out_met.items():
            for categorization, diff_key in [('_coarse', diff_coarse), ('_fine', diff_fine)]:
                kc = k + categorization
                if kc in metric_dict:
                    if diff_key in metric_dict[kc]:
                        metric_dict[kc][diff_key].append(v)
                    else:
                        metric_dict[kc][diff_key] = [v]
                        
    del feats, charts
    gc.collect()

# 5. Global Metrics Summarization
F1 = tp / (tp + 0.5 * (fp + fn)) if (tp + fp + fn) > 0 else 0
Prec = tp / (tp + fp) if (tp + fp) > 0 else 0
Recall = tp / (tp + fn) if (tp + fn) > 0 else 0   
MaxF1 = mtp / (mtp + 0.5 * (mfp + mfn)) if (mtp + mfp + mfn) > 0 else 0
MaxPrec = mtp / (mtp + mfp) if (mtp + mfp) > 0 else 0
MaxRecall = mtp / (mtp + mfn) if (mtp + mfn) > 0 else 0
MaxThresh = np.median(threshlist) if threshlist else 0

metric_dict['f1_avg'] = F1
metric_dict['prec_avg'] = Prec
metric_dict['recall_avg'] = Recall
metric_dict['max_f1_avg'] = MaxF1
metric_dict['max_prec_avg'] = MaxPrec
metric_dict['max_recall_avg'] = MaxRecall
metric_dict['max_thresh_avg'] = MaxThresh

print("\n--- Final Results Summary ---")
print(f'f1 avg: {F1:.4f}\nPrec avg: {Prec:.4f}\nRecall avg: {Recall:.4f}')
print(f'Max f1 avg: {MaxF1:.4f}\nMax Prec avg: {MaxPrec:.4f}\nMax Recall avg: {MaxRecall:.4f}')
print(f'Max Thresh median: {MaxThresh:.4f}')

# 6. Save metrics payload dictionary to pickle
torch_metrics_path = 'onset_torch_metrics.pkl'
with open(torch_metrics_path, 'wb') as f:
    pickle.dump(metric_dict, f)
print(f"\nSaved metrics payload dictionary to file: {torch_metrics_path}")

ITGPT_nohier_onset = metric_dict

Starting non-autoregressive one-shot validation pass over 24 files...

--- Final Results Summary ---
f1 avg: 0.7490
Prec avg: 0.7740
Recall avg: 0.7255
Max f1 avg: 0.7914
Max Prec avg: 0.7427
Max Recall avg: 0.8469
Max Thresh median: 0.3473

Saved metrics payload dictionary to file: onset_torch_metrics.pkl


In [11]:
from onset_cleaned import *
# 1. Recreate the precise configuration used during training
config = OnsetConfig(
    d_model=256,       # Default matching parser.add_argument("--d_model", default=256)
    n_enc_layers=4,    # Default matching parser.add_argument("--enc_layers", default=4)
    n_dec_layers=4     # Default matching parser.add_argument("--dec_layers", default=4)
)

# 2. Instantiate the model structural architecture
onset_model = OnsetModel(config)

# 3. Resolve the path to your best checkpoint file
# Matches: os.path.join(model_dir, f"{model_name}_best.pt")
model_dir = "trained_models"
model_name = "ddr_onset_v1"
checkpoint_path = os.path.join(model_dir, f"{model_name}_hier_clean_nodiag_best_f1.pt")

# 4. Load the weights state dictionary into the model
if os.path.exists(checkpoint_path):
    import numpy as np
    torch.serialization.add_safe_globals([np.core.multiarray.scalar])
    
    checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'), weights_only=False)
    
    # Extract the nested model weight weights dictionary based on the traceback fields
    if isinstance(checkpoint, dict) and 'model_state' in checkpoint:
        onset_model.load_state_dict(checkpoint['model_state'])
    elif isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        onset_model.load_state_dict(checkpoint['model_state_dict'])
    else:
        onset_model.load_state_dict(checkpoint)
        
    print(f"Successfully loaded best checkpoint weights from: {checkpoint_path}")
    if isinstance(checkpoint, dict) and 'epoch' in checkpoint:
        print(f"Checkpoint info -> Saved at Epoch: {checkpoint['epoch']} | Val PR AUC: {checkpoint.get('val_pr_auc', 'N/A')}")
else:
    raise FileNotFoundError(f"Could not find a checkpoint file at {checkpoint_path}.")

Successfully loaded best checkpoint weights from: trained_models/ddr_onset_v1_hier_clean_nodiag_best_f1.pt
Checkpoint info -> Saved at Epoch: 16 | Val PR AUC: 0.8621600235611906


In [13]:
import time
import gc

# Ensure model is ready and on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
onset_model.to(device)
onset_model.eval()

# 1. Initialize Tracking Structures matching your layout
metric_dict = {name: dict() for name in metrics}

vld_ds = get_dataset_fp_list(vld_fp)

tp, fp, fn = 0, 0, 0
mtp, mfp, mfn = 0, 0, 0
threshlist = []

with open(lbl_fp, 'rb') as f:
    labels = pickle.load(f)
enc_dict = label_to_vect_dict(labels)

# Initialize timing accumulators (in seconds)
total_io_time = 0.0
total_inference_time = 0.0
total_metrics_time = 0.0

# 2. Sequential Validation Pass over the dataset
print(f"Starting non-autoregressive one-shot validation pass over {len(vld_ds)} files...")
start_total_time = time.time()

for fpt in vld_ds:
    # --- Time I/O: Data Loading ---
    start_io = time.time()
    with open(fpt, 'rb') as f:
        charts = pickle.load(f)
        
    feats_fp = charts[3]  # Extract independent feature reference path
    with open(feats_fp, 'rb') as f:
        feats = pickle.load(f)
    total_io_time += (time.time() - start_io)
        
    # Step through every chart variant contained inside this entry
    for j in range(len(charts[0])):
        chart = [charts[i][j] for i in range(3)]
        
        diff_coarse = diff_dict[chart[1][0][-1]]
        diff_fine = chart[1][0][0]
        
        # Safely capture the scalar BPM sequence value before chart[1] is modified
        bpm_val = chart[1][0][1]

        # Resolve features from external file via window context ranges
        chart[0] = [make_onset_feature_context_range(feats, x[0], x[1]) for x in chart[0]]
        
        # Standardize features
        mean = np.mean(chart[0], axis=0)
        std = np.std(chart[0], axis=0)
        chart[0] = (np.array(chart[0]) - mean) / (std + 1e-7)
        
        chart[1] = [[a[0], a[1]] for a in chart[1]]
        placed_notes = np.array([enc_dict[i] for i in chart[2]])

        # ------------------------------------------------------------------
        # FIXED: One-Shot Score Extraction & Sigmoid Conversion
        # ------------------------------------------------------------------
        # Input shape expected by OnsetModel: (Batch=1, Time, Frames, Freq, Channels)
        feats_t = torch.tensor(chart[0], dtype=torch.float32).unsqueeze(0).to(device)
        
        bpm_t = torch.tensor([float(bpm_val)], dtype=torch.float32).to(device)
        diff_t = torch.tensor([int(diff_fine)], dtype=torch.long).to(device)
        
        # --- Time Inference Pass ---
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start_inf = time.time()
        
        with torch.no_grad():
            # Use predict_prob to ensure values are sigmoided probabilities bounded [0, 1]
            # or extract forward and apply torch.sigmoid directly.
            scores_t = onset_model.predict_prob(feats_t, bpm_t, diff_t)
            
            if device.type == 'cuda':
                torch.cuda.synchronize()
            total_inference_time += (time.time() - start_inf)
            
            # Squeeze batch dimension and transfer to CPU NumPy array: shape (T, 48)
            scores = scores_t.squeeze(0).cpu().numpy()
        # ------------------------------------------------------------------
        
        # --- Time Metrics Calculations ---
        start_metrics = time.time()
        
        # 3. Micro-level Multi-Label Metrics Extraction
        flat_placed = placed_notes.flatten()
        flat_scores = scores.flatten()
        
        prec, rec, threshes = precision_recall_curve(flat_placed, flat_scores)
        AUC = auc(rec, prec)
        
        # safely bound log_loss predictions using clip to avoid stability blowups at 0 or 1
        flat_scores_clipped = np.clip(flat_scores, 1e-7, 1 - 1e-7)
        loss = log_loss(flat_placed, flat_scores_clipped, labels=[0, 1])
        
        fd = prec + rec
        fd[np.where(fd == 0)] = 1
        fs = (2 * (prec * rec)) / fd
        MID = np.argmax(fs)
        
        # Prevent index errors if precision_recall_curve returns empty boundary arrays
        if len(threshes) > 0 and MID < len(threshes):
            MaxF1, MaxPrec, MaxRec, MaxThresh = fs[MID], prec[MID], rec[MID], threshes[MID]
        else:
            MaxF1, MaxPrec, MaxRec, MaxThresh = fs[MID], prec[MID], rec[MID], 0.5
            
        threshlist.append(MaxThresh)
        
        # Accumulate metrics based on dynamically found optimal threshold
        mtp += np.sum((flat_scores >= MaxThresh) & (flat_placed == 1))
        mfp += np.sum((flat_scores >= MaxThresh) & (flat_placed == 0))
        mfn += np.sum((flat_scores < MaxThresh) & (flat_placed == 1))

        chart_tp = np.sum((flat_scores >= 0.5) & (flat_placed == 1))
        chart_fp = np.sum((flat_scores >= 0.5) & (flat_placed == 0))
        chart_fn = np.sum((flat_scores < 0.5) & (flat_placed == 1))
        
        # Global accumulation trackers
        tp += chart_tp
        fp += chart_fp
        fn += chart_fn

        # Calculate exact chart-level metrics via the counting method
        F1 = chart_tp / (chart_tp + 0.5 * (chart_fp + chart_fn)) if (chart_tp + chart_fp + chart_fn) > 0 else 0
        Prec = chart_tp / (chart_tp + chart_fp) if (chart_tp + chart_fp) > 0 else 0
        Recall = chart_tp / (chart_tp + chart_fn) if (chart_tp + chart_fn) > 0 else 0
        out_met = {
            'log_loss': loss, 'f1': F1, 'auc': AUC, 'prec': Prec, 'recall': Recall, 
            'max_f1': MaxF1, 'max_prec': MaxPrec, 'max_recall': MaxRec, 'max_thresh': MaxThresh
        }
        
        # Populate categorized nested dictionaries
        for k, v in out_met.items():
            for categorization, diff_key in [('_coarse', diff_coarse), ('_fine', diff_fine)]:
                kc = k + categorization
                if kc in metric_dict:
                    if diff_key in metric_dict[kc]:
                        metric_dict[kc][diff_key].append(v)
                    else:
                        metric_dict[kc][diff_key] = [v]
        
        total_metrics_time += (time.time() - start_metrics)
                        
    del feats, charts
    gc.collect()

# 5. Global Metrics Summarization
F1 = tp / (tp + 0.5 * (fp + fn)) if (tp + fp + fn) > 0 else 0
Prec = tp / (tp + fp) if (tp + fp) > 0 else 0
Recall = tp / (tp + fn) if (tp + fn) > 0 else 0   
MaxF1 = mtp / (mtp + 0.5 * (mfp + mfn)) if (mtp + mfp + mfn) > 0 else 0
MaxPrec = mtp / (mtp + mfp) if (mtp + mfp) > 0 else 0
MaxRecall = mtp / (mtp + mfn) if (mtp + mfn) > 0 else 0
MaxThresh = np.median(threshlist) if threshlist else 0

metric_dict['f1_avg'] = F1
metric_dict['prec_avg'] = Prec
metric_dict['recall_avg'] = Recall
metric_dict['max_f1_avg'] = MaxF1
metric_dict['max_prec_avg'] = MaxPrec
metric_dict['max_recall_avg'] = MaxRecall
metric_dict['max_thresh_avg'] = MaxThresh

end_total_time = time.time()
elapsed_total = end_total_time - start_total_time

print("\n--- Final Results Summary ---")
print(f'f1 avg: {F1:.4f}\nPrec avg: {Prec:.4f}\nRecall avg: {Recall:.4f}')
print(f'Max f1 avg: {MaxF1:.4f}\nMax Prec avg: {MaxPrec:.4f}\nMax Recall avg: {MaxRecall:.4f}')
print(f'Max Thresh median: {MaxThresh:.4f}')

print("\n--- Benchmark Timing Breakdown ---")
print(f"Total loop time:       {elapsed_total:.2f}s")
print(f"├─ File I/O time:      {total_io_time:.2f}s ({(total_io_time/elapsed_total)*100:.1f}%)")
print(f"├─ Inference time:     {total_inference_time:.2f}s ({(total_inference_time/elapsed_total)*100:.1f}%)")
print(f"└─ Metric/Eval time:   {total_metrics_time:.2f}s ({(total_metrics_time/elapsed_total)*100:.1f}%)")

# 6. Save metrics payload dictionary to pickle
torch_metrics_path = 'onset_torch_metrics.pkl'
with open(torch_metrics_path, 'wb') as f:
    pickle.dump(metric_dict, f)
print(f"\nSaved metrics payload dictionary to file: {torch_metrics_path}")

ITGPT_nodiag_onset = metric_dict

Starting non-autoregressive one-shot validation pass over 24 files...

--- Final Results Summary ---
f1 avg: 0.7701
Prec avg: 0.7454
Recall avg: 0.7964
Max f1 avg: 0.7998
Max Prec avg: 0.7545
Max Recall avg: 0.8509
Max Thresh median: 0.4352

--- Benchmark Timing Breakdown ---
Total loop time:       4.73s
├─ File I/O time:      0.05s (1.0%)
├─ Inference time:     1.18s (24.9%)
└─ Metric/Eval time:   0.28s (5.9%)

Saved metrics payload dictionary to file: onset_torch_metrics.pkl


In [19]:
# Create names mapping and prepare dictionaries
# keras_metric_dict should contain keys like 'ddc' and 'onset' (DDCL)
model_names = ['DDC', 'DDCL', 'GOCT', 'ITGPT', 'ITGPT (No Hier.)', 'ITGPT (No Diag)']
dicts = [
    keras_metric_dict.get('ddc', {}),
    keras_metric_dict.get('ddcl', {}), # DDCL 
    goct_metric_dict,
    ITGPT_onset,
    ITGPT_nohier_onset,
    ITGPT_nodiag_onset
]

metrics_to_show = ['f1_avg', 'prec_avg', 'recall_avg', 'max_f1_avg', 'max_prec_avg', 'max_recall_avg']

metric_name_list = [
    'F1 Score',
    'Prec.',
    'Recall',
    'Max F1 Score',
    'Max Prec.',
    'Max Recall'
]

# Build the comparison rows
table_rows = []
for met, label in zip(metrics_to_show, metric_name_list):
    row = [label]
    for d in dicts:
        # If the metric is a scalar directly
        if isinstance(d.get(met), (int, float)):
            row.append(f"{d[met]:.4f}")
        # Otherwise if it requires averaging (fallback standard safe handle)
        elif met in d and isinstance(d[met], dict):
            row.append(f"{np.mean(list(d[met].values())):.4f}")
        else:
            row.append("N/A")
    table_rows.append(row)

print("--- Overall Model Performance Comparison ---")
print(tabulate(table_rows, headers=['Metric'] + model_names, tablefmt='grid'))

--- Overall Model Performance Comparison ---
+--------------+--------+--------+--------+---------+--------------------+-------------------+
| Metric       |    DDC |   DDCL | GOCT   |   ITGPT |   ITGPT (No Hier.) |   ITGPT (No Diag) |
+==============+========+========+========+=========+====================+===================+
| F1 Score     | 0.5006 | 0.7033 | N/A    |  0.7801 |             0.749  |            0.7701 |
+--------------+--------+--------+--------+---------+--------------------+-------------------+
| Prec.        | 0.703  | 0.7239 | N/A    |  0.7538 |             0.774  |            0.7454 |
+--------------+--------+--------+--------+---------+--------------------+-------------------+
| Recall       | 0.3887 | 0.6838 | N/A    |  0.8084 |             0.7255 |            0.7964 |
+--------------+--------+--------+--------+---------+--------------------+-------------------+
| Max F1 Score | 0.7317 | 0.7598 | 0.7754 |  0.8022 |             0.7914 |            0.7998 |
+----

In [20]:
# Provided variables
headers = ['\\textbf{Metric}', '\\textbf{DDC}', '\\textbf{DDCL}', '\\textbf{GOCT}', '\\textbf{ITGPT}', '\\textbf{ITGPT (NH)}', '\\textbf{ITGPT (ND)}']

metrics_to_show = ['f1_avg','prec_avg','recall_avg','max_f1_avg','max_prec_avg','max_recall_avg']
metrics_to_avg = ['f1_coarse','prec_coarse','recall_coarse','max_f1_coarse','max_prec_coarse','max_recall_coarse','log_loss_coarse', 'auc_coarse']

metric_name_list = [
    'F1Score',
    'Prec.',
    'Recall',
    'Max F1Score',
    'Max Prec.',
    'Max Recall',
    'F1Score\\textsubscript{\\textbf{cht}}',
    'Prec\\textsubscript{\\textbf{cht}}',
    'Recall\\textsubscript{\\textbf{cht}}',
    'Max F1\\textsubscript{\\textbf{cht}}',
    'Max Prec.\\textsubscript{\\textbf{cht}}',
    'Max Rec.\\textsubscript{\\textbf{cht}}',
    'CE\\textsubscript{\\textbf{cht}}', 
    'AUC\\textsubscript{\\textbf{cht}}', 
]

metric_name_dict = dict(zip(metrics_to_show + metrics_to_avg, metric_name_list))

# Create rows of data
rows = []
row_labels = []

for met in metrics_to_show:
    row = [np.mean(dic.get(met, [np.nan])) for dic in dicts]
    rows.append(row)
    row_labels.append(met)

for met in metrics_to_avg:
    row = [np.mean(sum(dic.get(met, {}).values(), [])) for dic in dicts]
    rows.append(row)
    row_labels.append(met)

# Create DataFrame with proper column names
df = pd.DataFrame(rows, columns=headers[1:])
df.insert(0, headers[0], row_labels)

# Custom LaTeX table with bolding and hlines
latex_rows = []

# Add header
header_line = ' & '.join(headers) + ' \\\\ \\hline'
latex_rows.append(header_line)

# Format each row with bolding logic
for idx, row in df.iterrows():
    metric_key = row[headers[0]]
    values = row[1:].astype(float).values

    is_log_loss = 'log_loss' in metric_key
    target_func = np.nanargmin if is_log_loss else np.nanargmax
    target_idx = target_func(values)

    formatted_values = []
    for i, val in enumerate(values):
        val_str = f"{val:.4f}" if not np.isnan(val) else "-"
        if i == target_idx and val_str != "-":
            val_str = f"\\textbf{{{val_str}}}"
        formatted_values.append(val_str)

    metric_latex_name = metric_name_dict.get(metric_key.replace('_chart_avg', ''), metric_key)
    latex_rows.append(f"{metric_latex_name} & " + ' & '.join(formatted_values) + " \\\\ \\hline")

# Final LaTeX table string
latex_table = "\\begin{tabular}{|l" + "|c" * (len(headers) - 1) + "|}\n\\hline\n"
latex_table += '\n'.join(latex_rows)
latex_table += "\n\\end{tabular}"

print(latex_table)


\begin{tabular}{|l|c|c|c|c|c|c|}
\hline
\textbf{Metric} & \textbf{DDC} & \textbf{DDCL} & \textbf{GOCT} & \textbf{ITGPT} & \textbf{ITGPT (NH)} & \textbf{ITGPT (ND)} \\ \hline
F1Score & 0.5006 & 0.7033 & - & \textbf{0.7801} & 0.7490 & 0.7701 \\ \hline
Prec. & 0.7030 & 0.7239 & - & 0.7538 & \textbf{0.7740} & 0.7454 \\ \hline
Recall & 0.3887 & 0.6838 & - & \textbf{0.8084} & 0.7255 & 0.7964 \\ \hline
Max F1Score & 0.7317 & 0.7598 & 0.7754 & \textbf{0.8022} & 0.7914 & 0.7998 \\ \hline
Max Prec. & 0.6606 & 0.6995 & 0.7500 & \textbf{0.7621} & 0.7427 & 0.7545 \\ \hline
Max Recall & 0.8198 & 0.8313 & 0.8027 & 0.8467 & 0.8469 & \textbf{0.8509} \\ \hline
F1Score\textsubscript{\textbf{cht}} & 0.4850 & 0.6543 & - & 0.7348 & 0.7039 & \textbf{0.7352} \\ \hline
Prec\textsubscript{\textbf{cht}} & 0.6721 & 0.6818 & - & 0.7282 & \textbf{0.7394} & 0.7270 \\ \hline
Recall\textsubscript{\textbf{cht}} & 0.4636 & 0.6848 & - & 0.7604 & 0.6935 & \textbf{0.7640} \\ \hline
Max F1\textsubscript{\textbf{cht}} & 0.65

/home/miguelomalley/pythonl/lib/python3.10/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/miguelomalley/pythonl/lib/python3.10/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [22]:
import pandas as pd
import numpy as np

# Stratified breakdown settings matching original script
metrics_to_breakdown = [
    'f1_coarse', 'prec_coarse', 'recall_coarse',
    'max_f1_coarse', 'max_prec_coarse', 'max_recall_coarse',
    'log_loss_coarse', 'auc_coarse'
]

metric_labels = {
    'f1_coarse': 'F1 Score',
    'prec_coarse': 'Precision',
    'recall_coarse': 'Recall',
    'max_f1_coarse': 'Max F1 Score',
    'max_prec_coarse': 'Max Precision',
    'max_recall_coarse': 'Max Recall',
    'log_loss_coarse': 'Log Loss',
    'auc_coarse': 'PR AUC'
}

rev_diff_dict = {0: 'Beginner', 1: 'Easy', 2: 'Medium', 3: 'Hard', 4: 'Challenge'}

rows = []
index = []

# Collect data for the MultiIndex DataFrame
for met in metrics_to_breakdown:
    # Safely find available difficulty keys from one of the filled dictionaries
    available_diffs = sorted(metric_dict.get(met, {}).keys())
    for diff in available_diffs:
        row = []
        for d in dicts:
            metric_data = d.get(met, {}).get(diff, [])
            # Handle if it is a list of entries or a singular pre-aggregated value
            if isinstance(metric_data, list) and len(metric_data) > 0:
                row.append(np.mean(metric_data))
            elif isinstance(metric_data, (int, float)):
                row.append(float(metric_data))
            else:
                row.append(np.nan)
        rows.append(row)
        index.append((metric_labels[met], diff))

# Construct MultiIndex DataFrame
multi_index = pd.MultiIndex.from_tuples(index, names=["Metric", "Difficulty"])
df_grouped = pd.DataFrame(rows, index=multi_index, columns=model_names)

# Custom LaTeX formatting loop identical to original script workflow
latex_lines = []
col_format = '|c|l|' + 'c|' * len(model_names)
latex_lines.append(f"\\begin{{tabular}}{{{col_format}}}")
latex_lines.append("\\hline")
header_row = ['Metric', 'Difficulty'] + model_names
latex_lines.append(' & '.join(header_row) + ' \\\\ \\hline')

prev_metric = None
for (metric, diff), row in df_grouped.iterrows():
    values = row.values.astype(float)
    
    # Identify best score direction (minimize Log Loss, maximize everything else)
    is_log_loss = 'Log Loss' in metric
    if np.all(np.isnan(values)):
        target_idx = -1
    else:
        target_idx = np.nanargmin(values) if is_log_loss else np.nanargmax(values)

    formatted_values = []
    for i, val in enumerate(values):
        if np.isnan(val):
            val_str = "-"
        else:
            val_str = f"{val:.6f}"
            if i == target_idx:
                val_str = f"\\textbf{{{val_str}}}"
        formatted_values.append(val_str)

    # Multirow configuration for structural neatness
    metric_label = f"\\multirow{{5}}{{*}}{{{metric}}}" if metric != prev_metric else ''
    prev_metric = metric

    line = f"{metric_label} & {rev_diff_dict.get(diff, diff)} & " + ' & '.join(formatted_values) + " \\\\ "
    # Add a horizontal line block separation when shifting metrics
    if metric_label == '' and diff == list(rev_diff_dict.keys())[-1]:
        line += "\\hline"
    latex_lines.append(line)

latex_lines.append("\\end{tabular}")

latex_output = '\n'.join(latex_lines)
print("\n--- Formatted LaTeX Table Output ---")
print(latex_output)


--- Formatted LaTeX Table Output ---
\begin{tabular}{|c|l|c|c|c|c|c|c|}
\hline
Metric & Difficulty & DDC & DDCL & GOCT & ITGPT & ITGPT (No Hier.) & ITGPT (No Diag) \\ \hline
\multirow{5}{*}{F1 Score} & Beginner & 0.237887 & 0.345704 & - & 0.555398 & 0.472772 & \textbf{0.586783} \\ 
 & Easy & 0.451751 & 0.714626 & - & 0.762072 & 0.748279 & \textbf{0.769704} \\ 
 & Medium & 0.566833 & 0.692552 & - & \textbf{0.742250} & 0.710226 & 0.736506 \\ 
 & Hard & 0.578538 & 0.704101 & - & \textbf{0.748957} & 0.743579 & 0.747109 \\ 
 & Challenge & 0.515150 & 0.714488 & - & \textbf{0.787855} & 0.757281 & 0.776051 \\ \hline
\multirow{5}{*}{Precision} & Beginner & 0.163650 & 0.235650 & - & \textbf{0.557812} & 0.499722 & 0.555460 \\ 
 & Easy & 0.415487 & 0.700121 & - & 0.743895 & \textbf{0.760946} & 0.748669 \\ 
 & Medium & 0.673378 & 0.735903 & - & 0.765953 & \textbf{0.773669} & 0.753717 \\ 
 & Hard & \textbf{0.817504} & 0.750258 & - & 0.725445 & 0.775813 & 0.738284 \\ 
 & Challenge & \textbf{0.911447

In [29]:
import numpy as np
import pandas as pd

# Metric keys and labels
metrics_to_show = [
    'f1_fine', 'prec_fine', 'recall_fine',
    'max_f1_fine', 'max_prec_fine', 'max_recall_fine',
    'log_loss_fine', 'auc_fine'
]

metric_labels = {
    'f1_fine': 'F1 Score',
    'prec_fine': 'Precision',
    'recall_fine': 'Recall',
    'max_f1_fine': 'Max F1',
    'max_prec_fine': 'Max Precision',
    'max_recall_fine': 'Max Recall',
    'log_loss_fine': 'Log Loss',
    'auc_fine': 'AUC'
}

model_names = ['DDC', 'DDCL', 'GOCT', 'ITGPT', 'ITGPT (NH)', 'ITGPT (ND)']

# Collect data and build MultiIndex
rows = []
index = []

for met in metrics_to_show:
    # Safely find all unique difficulties across all models for this metric
    all_diffs = set()
    for dic in dicts:
        all_diffs.update(dic.get(met, {}).keys())
    
    for diff in sorted(list(all_diffs)):
        row = []
        for dic in dicts:
            # Safely navigate nested dictionaries; fallback to an empty list
            metric_data = dic.get(met, {}).get(diff, [])
            # Calculate mean if data exists, otherwise insert NaN
            row.append(np.mean(metric_data) if len(metric_data) > 0 else np.nan)
            
        rows.append(row)
        index.append((metric_labels.get(met, met), diff))

# Build MultiIndex DataFrame (only if rows were added)
if rows:
    multi_index = pd.MultiIndex.from_tuples(index, names=["Metric", "Difficulty"])
    df_grouped = pd.DataFrame(rows, index=multi_index, columns=model_names)
else:
    df_grouped = pd.DataFrame(columns=model_names)

# --- Custom LaTeX formatting ---
latex_lines = []

# Header
col_format = '|c|l|' + 'c|' * len(model_names)
latex_lines.append(f"\\begin{{tabular}}{{{col_format}}}")
latex_lines.append("\\hline")
header_row = ['Metric', 'Difficulty'] + model_names
latex_lines.append(' & '.join(header_row) + ' \\\\ \\hline')

# Data rows with bolding
prev_metric = None
for idx, ((metric, diff), row) in enumerate(df_grouped.iterrows()):
    values = row.values.astype(float)
    
    is_log_loss = 'Log Loss' in metric
    
    # Filter out NaNs when determining the min/max index
    valid_mask = ~np.isnan(values)
    if np.any(valid_mask):
        if is_log_loss:
            target_idx = np.where(valid_mask, values, np.inf).argmin()
        else:
            target_idx = np.where(valid_mask, values, -np.inf).argmax()
    else:
        target_idx = -1

    formatted_values = []
    for i, val in enumerate(values):
        if np.isnan(val):
            val_str = "-"
        else:
            val_str = f"{val:.6f}"
            if i == target_idx:
                val_str = f"\\textbf{{{val_str}}}"
        formatted_values.append(val_str)

    # Only show metric name for the first row of each group
    metric_label = "\multirow{12}{*}{" + metric + "}" if metric != prev_metric else ''
    prev_metric = metric

    line = f"{metric_label} & {diff} & " + ' & '.join(formatted_values) + " \\\\ "
    
    # Check if this is the last item in the current metric group or last item overall
    if idx == len(df_grouped) - 1 or df_grouped.index[idx + 1][0] != metric:
        line += "\\hline"
        
    latex_lines.append(line)

latex_lines.append("\\end{tabular}")

# Final output
latex_output = '\n'.join(latex_lines)
print(latex_output)

\begin{tabular}{|c|l|c|c|c|c|c|c|}
\hline
Metric & Difficulty & DDC & DDCL & GOCT & ITGPT & ITGPT (NH) & ITGPT (ND) \\ \hline
\multirow{12}{*}{F1 Score} & 1 & 0.237887 & 0.345704 & - & 0.555398 & 0.472772 & \textbf{0.586783} \\ 
 & 3 & 0.449043 & 0.702762 & - & 0.750893 & 0.743661 & \textbf{0.759200} \\ 
 & 4 & 0.463938 & 0.768015 & - & 0.812379 & 0.769064 & \textbf{0.816972} \\ 
 & 5 & 0.579838 & 0.706815 & - & 0.738970 & \textbf{0.739258} & 0.735267 \\ 
 & 6 & 0.518100 & 0.627068 & - & \textbf{0.689448} & 0.649349 & 0.685083 \\ 
 & 7 & 0.593889 & 0.734263 & - & \textbf{0.800521} & 0.722717 & 0.789994 \\ 
 & 8 & 0.599181 & 0.677430 & - & \textbf{0.741127} & 0.731411 & 0.730693 \\ 
 & 9 & 0.592145 & 0.715236 & - & 0.761263 & 0.770049 & \textbf{0.778120} \\ 
 & 10 & 0.636111 & 0.652165 & - & \textbf{0.708790} & 0.679721 & 0.691311 \\ 
 & 11 & 0.520058 & 0.718407 & - & \textbf{0.792486} & 0.768294 & 0.789467 \\ 
 & 12 & 0.503756 & 0.699257 & - & \textbf{0.785233} & 0.743668 & 0.762625 \\